In [7]:
# !pip install -q openai langchain-openai langchain-community faiss-cpu python-dotenv gradio
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [8]:
# 환경 설정
import os
from dotenv import load_dotenv
import pandas as pd
import json
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

오늘은 프롬프트 엔지니어링 다룰 예정
- 참고: https://developers.openai.com/api/docs/guides/prompt-engineering
- 프롬프트 엔지니어링(Prompt Engineering)은 LLM의 파라미터를 수정하지 않고, 입력 텍스트(프롬프트)의 설계만으로 모델의 출력 품질을 제어·향상시키는 기술 분야.


## 프롬프트 엔지니어링 기초

In [4]:
# 좋은 프롬프트?
  # 지시
  # context
  # input
  # output format

In [6]:
bad_prompt = 'AI에 대해 알려줘'
llm.invoke([HumanMessage(content=bad_prompt)])

AIMessage(content='AI(인공지능)는 기계나 컴퓨터가 인간의 지능을 모방하여 학습, 문제 해결, 의사 결정, 언어 이해, 이미지 인식 등을 수행할 수 있도록 하는 기술입니다. AI는 크게 두 가지 범주로 나눌 수 있습니다:\n\n1. **좁은 인공지능(Narrow AI)**: 특정 작업에 특화된 AI로, 예를 들어 음성 인식, 이미지 인식, 게임 플레이 등 특정 분야에서 높은 성능을 발휘합니다. 가장 많은 응용 사례가 이 범주에 해당합니다.\n\n2. **인공지능 일반(General AI)**: 인간과 같은 수준의 지능을 가진 AI로, 다양한 작업을 수행할 수 있는 능력을 가진 것입니다. 현재는 이 수준의 AI는 존재하지 않으며, 연구의 주제로 남아 있습니다.\n\nAI 기술의 주요 구성 요소에는 머신러닝(기계 학습), 딥러닝(심층 학습), 자연어 처리(NLP), 컴퓨터 비전 등이 있습니다. \n\n- **머신러닝**: 데이터에서 패턴을 학습하고 예측을 수행하는 알고리즘입니다.\n- **딥러닝**: 인공 신경망을 기반으로 한 하위 영역으로, 특히 대량의 데이터를 처리하는 데 적합합니다.\n- **자연어 처리**: 인간의 언어를 이해하고 생성하는 기술로, 챗봇, 번역기 등에서 사용됩니다.\n- **컴퓨터 비전**: 이미지를 분석하고 이해하는 기술로, 자율주행차, 얼굴 인식 등에 활용됩니다.\n\nAI는 의료, 금융, 제조, 교육, 운송 등 다양한 분야에서 활용되고 있으며, 인류의 삶을 향상시키는 데 기여하고 있습니다. 그러나 윤리적 문제나 일자리 대체 등에 대한 우려도 존재하여, 적절한 규제와 사회적 논의가 중요한 시점에 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 411, 'prompt_tokens': 12, 'total_tokens': 423, 'completion_tokens_details': {'accept

In [7]:
good_prompt = """
[지시] 아래 주제에 대해 초보자용 설명을 작성해주세요.
[컨텍스트] 기업 신입서원 대상 AI 교육 자료입니다.
[입력] 대규모 언어모델(LLM)의 작동 원리
[출력 형식] 3문장 이내로, 비유를 하나 포함하세요
"""
llm.invoke([HumanMessage(content=good_prompt)])

AIMessage(content='대규모 언어 모델(LLM)은 방대한 양의 텍스트 데이터를 학습하여 인간처럼 언어를 이해하고 생성하는 인공지능 시스템입니다. 이를 비유하자면, LLM은 수천 권의 책을 읽고 그 내용을 기억하는 거대한 도서관과 같습니다. 이 도서관은 질문에 맞는 정보를 빠르게 찾거나 새로운 이야기를 만들어내는 데 도움을 줍니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 83, 'total_tokens': 176, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DMu0p3w2Czg8bolF6WpW5pwT5Z7RD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1f8e-1366-74f3-89d7-de6a79ae4c54-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 93, 'total_tokens': 176, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_tok

In [18]:
def build_prompt(instructions, context="", input_data="", output_format=""):
  parts = []
  parts.append(f"[지시] {instructions}")
  if context:
    parts.append(f"[컨텍스트] {context}")
  if input_data:
    parts.append(f"[입력] {input_data}")
  if output_format:
    parts.append(f"[출력 형식] {output_format}")

  return '\n'.join(parts)



In [19]:
prompt = build_prompt(instructions = '다음 텍스트의 감정을 분석하세요',
                      context = '고객 리뷰 분석 시스템입니다.',
                      input_data ="배송은 빨랐는데 포장이 엉망입니다.",
                      output_format=  '감정 : [긍정/부정/혼합], 이유: [한 줄 설명]'
                      )

In [20]:
print(prompt)

[지시] 다음 텍스트의 감정을 분석하세요
[컨텍스트] 고객 리뷰 분석 시스템입니다.
[입력] 배송은 빨랐는데 포장이 엉망입니다.
[출력 형식] 감정 : [긍정/부정/혼합], 이유: [한 줄 설명]


In [21]:
llm.invoke([HumanMessage(content=prompt)])

AIMessage(content='감정 : 혼합, 이유: 빠른 배송에 대한 긍정적인 평가와 포장 문제에 대한 부정적인 평가가 혼재되어 있음.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 76, 'total_tokens': 110, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DMu5xbju5ow5AOcI8dg9wGR0oPt65', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1f92-f158-7222-b29d-bb388616c6ec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 34, 'total_tokens': 110, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [29]:
input_data = """도널드 트럼프 미국 대통령은 23일(현지시간) 이란 최고 지도부와 대화 중이며 이란의 핵능력 포기 등 15가지를 합의한 상태라고 말했다. 또 미국과 이란이 중동 사태 해결을 위해 최근 이틀간 대화를 나눠 왔다며 이란 발전소ㆍ인프라 공격을 5일간 유예한다고 했다.

지난달 28일 미ㆍ이스라엘의 공습으로 시작된 이란 전쟁이 이날로 24일째를 맞은 가운데 트럼프 행정부가 이란과 협상에 나섰다고 공개한 것은 이날이 처음이다. 다만 이란은 미국과 어떤 협상이나 대화도 없었다며 이를 부인했다.

이스라엘 “국익 수호 합의로 목표 달성 기회”
이스라엘 베냐민 네타냐후 총리는 트럼프 대통령과의 통화 사실을 공개하면서 그의 ‘협상’ 발언을 간접적으로 뒷받침했다. 네타냐후 총리는 “이스라엘의 핵심 이익을 수호하는 합의를 통해 전쟁 목표 달성의 기회가 있다고 믿는다”고 말했다. 사실 여부는 공식 확인되지 않고 있는 가운데 트럼프 대통령 말대로 물밑 협상이 벌어지고 있다면 그 결과가 확전 여부를 가르는 중대 분수령이 될 것으로 관측된다.

트럼프 대통령은 이날 테네시주 멤피스에서 열린 ‘멤피스 안전 태스크포스 원탁회의’ 행사 연설에서 “이란과는 오랫동안 협상을 해 왔고 이번에는 그들이 진지하게 임하고 있다”며 “그들은 합의를 원하고 있고, 우리는 이를 성사시킬 것”이라고 말했다. 이어 “이제 이란은 미국과 우리 동맹국들에 대한 위협을 끝낼 마지막 기회를 한 번 더 얻었고, 우리는 그들이 이 기회를 잡기를 바란다”고 말했다. 그러면서 “5일간의 시간을 주고 결과를 지켜볼 것이다. 이 기간이 끝날 무렵에는 모두에게 매우 좋은 결과가 나올 수 있다고 본다”고 말했다.

트럼프 ‘48시간 시한’ 12시간 남기고 철회
앞서 트럼프 대통령은 이날 플로리다주 웨스트팜비치에서 대통령 전용기에 오르기 전 기자들과 만나서는 스티브 위트코프 대통령 중동 특사와 자신의 사위인 재러드 쿠슈너가 이란 측과 매우 실질적인 협상을 진행했다고 말했다. 트럼프 대통령은 “우리도 합의를 원한다. 오늘 우리는 아마도 전화로 만나게 될 것”이라며 “아주 가까운 시일 내에 직접 만날 것이다. 5일간의 기간을 두고 있다”고 말했다. [출처:중앙일보] https://www.joongang.co.kr/article/25414242"""

In [30]:
# 실습
# build_prompt 함수를 이용하여 '뉴스 기사 요약기'를 만들어 보세요.

prompt = build_prompt(instructions = '다음 텍스트를 요약하세요',
                      context = '뉴스 기사 요약 시스템입니다.',
                      input_data = input_data,
                      output_format=  '요약 : [한줄 설명], 주요 키워드: [키워드 최대 5자, 쉽표로 구분]'
                      )

In [31]:
llm.invoke([HumanMessage(content=prompt)])

AIMessage(content='요약 : 트럼프 대통령은 이란과의 협상 진전을 언급하며, 이란의 핵능력 포기와 중동 사태 해결을 위한 대화가 진행 중이라고 밝혔다. 이란은 이러한 협상 사실을 부인하고 있다.  \n주요 키워드: 트럼프, 이란, 협상, 중동, 핵능력', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 711, 'total_tokens': 791, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_64a7de84d3', 'id': 'chatcmpl-DMuAm4AV3QCUwJs6nNqebmu7P6yEA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d1f97-7e46-70d0-ab15-89860e88fe15-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 711, 'output_tokens': 80, 'total_tokens': 791, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [ ]:
# 모델이 예측해야할 것이 적을수록, 명확할 수록 랜덤성이 적어짐.

# AI에 대해 알려줘-> 어떻게, 어디부터 어디까지, 얼만큼, 누구한테 ..(기준이 없으면 학습데이터 전체 범위..)
# 수량, 대상, 제외조건, 품질 기준 등.

In [32]:
bad_prompt = '파이썬에 대해 알려주세요'
good_prompt = """
파이썬 프로그래밍 언어에 대해서 설명해주세요.

조건:
 - 대상: 프로그래밍 경험이 없는 일반인
 - 분량:3문장
 - 포함:파이썬의 주요 용도 1가지
 - 제외:코드 예시, 기술 용어
 - 문체 : 친근한 대화체
"""


## 역할 기반 프롬프팅

### 역할 프롬프팅
- 참고논문
  - Zero-Shot Reasoning with Role-Play Prompting
: https://arxiv.org/pdf/2308.07702
  - ExpertPrompting: https://arxiv.org/pdf/2305.14688
  - 모호한 페르소나 설정과 성능의 연관성: https://arxiv.org/pdf/2311.10054

- **제로샷 프롬프팅**이란 모델에게 해결해야 할 작업에 대한 예시(Example)를 단 하나도 제공하지 않고, 오직 **지시문(Instruction)**만으로 결과물을 생성하도록 하는 방법.

In [35]:
# 역할을 부여하면 특정 답변을 낼 확률을 높임.
question = "블록 체인 기술이 금융 산업에 미치는 영향은?"

roles = {
    "투자 전문가": "당신은 월스트리트에서 20년 경력의 투자 전문가입니다. 항상 투자 관점에서 생각하고, 구체적인 사례를 알려주세요.",
    "기술 개발자": "당신은 블록체인 코어 개발자입니다. 기술적 관점에서 설명하세요.",
    "규제 전문가": "당신은 금융감독원 출신 규제 전문가입니다. 법적/제도적 관점에서 설명해주세요.",
}

for role_name, role_msg in roles.items():
  print(f"roles: {role_name}")
  result = llm.invoke([SystemMessage(content = role_msg), HumanMessage(content=question)]).content
  print(f'response: {result[:200]}')
  print()

roles: 투자 전문가
response: 블록체인 기술은 금융 산업에 큰 영향을 미치고 있으며, 이는 여러 측면에서 나타나고 있습니다. 아래는 블록체인 기술이 금융 산업에 미치는 주요 영향과 몇 가지 구체적인 사례를 설명합니다.

### 1. **투명성 및 보안성**
블록체인은 거래 기록을 모든 참여자가 공유하는 분산형 장부 형태로 저장합니다. 이는 거래의 투명성을 높이고, 불법 행위를 방지하는 

roles: 기술 개발자
response: 블록체인 기술은 금융 산업에 여러 가지 중요한 영향을 미치고 있습니다. 아래는 그 주요 영향을 기술적 관점에서 설명한 것입니다.

1. **분산 원장 기술(Distributed Ledger Technology, DLT)**:
   - 블록체인은 분산된 원장을 사용하여 거래 데이터를 기록합니다. 이로 인해 중앙 기관(예: 은행)의 개입 없이도 모든 참여자가 

roles: 규제 전문가
response: 블록체인 기술은 금융 산업에 여러 가지 중요한 영향을 미치고 있습니다. 법적 및 제도적 관점에서 살펴보면 다음과 같은 요소들이 있습니다.

1. **투명성 및 추적 가능성**: 블록체인 기술은 모든 거래 기록을 분산된 장부에 저장하므로, 거래의 투명성을 높이고 사후 검증이 용이합니다. 이는 자금 세탁이나 금융 범죄를 방지하는 데 중요한 역할을 하며, 규제 



In [50]:
class RoleBasedAssistant:
  def __init__(self, role_name, system_prompt):
    self.role_name = role_name
    self.system_prompt = system_prompt
    self.history = [SystemMessage(content=system_prompt)]

  def ask(self, question):
    self.history.append(HumanMessage(content=question))
    response = llm.invoke(self.history)
    answer = response.content
    self.history.append(AIMessage(content=answer))
    return answer

  def reset(self):
    self.history = [SystemMessage(content=self.system_prompt)]

# 객체도 즉시 새로 생성
reviewer = RoleBasedAssistant(
    role_name = '시니어 코드 리뷰어',
    system_prompt = """당신은 10년 경력의 python 개발자이자 코드 리뷰어입니다.
    규칙:
    - 코드의 장단점을 균형있게 평가해주세요
    - 개선 제안은 항상 코드 예시와 함께 주세요
    - 주니어 개발자를 격려하는 톤을 유지해주세요"""
)

In [40]:
reviewer = RoleBasedAssistant(
    role_name = '시니어 코드 리뷰어',
    system_prompt = """당신은 10년 경력의 python 개발자이자 코드 리뷰어입니다.
    규칙:
    - 코드의 장단점을 균형있게 평가해주세요
    - 개선 제안은 항상 코드 예시와 함께 주세요
    - 주니어 개발자를 격려하는 톤을 유지해주세요"""
)

In [46]:
sample_code = """
def calc(x):
    r = []
    for i in x:
        if i > 0:
            r.append(i*2)
    return r
"""

# 이제 정상적으로 호출됩니다.
print(reviewer.ask(f"이 코드를 리뷰해주세요:\n{sample_code}"))

TypeError: 'builtin_function_or_method' object is not subscriptable

In [47]:
# 실습 RoleBasedAssistant 클래스를 활용해서
# CEO, CTO, CFO 3명의 전문가를 생성하여 "AI 도입 전략"에 대해 각 전문가의 의견을 받아보세요.
# 별도의 "사회자" assistant를 만들어서 3명의 응답을 하나의 메시지로 전달해주세요.

In [ ]:
reviewer = RoleBasedAssistant(
    role_name = 'CEO',
    system_prompt = """당신은 10년 경력의 python 개발자이자 코드 리뷰어입니다.
    규칙:
    - 코드의 장단점을 균형있게 평가해주세요
    - 개선 제안은 항상 코드 예시와 함께 주세요
    - 주니어 개발자를 격려하는 톤을 유지해주세요"""
)

In [9]:
class RoleBasedAssistant:
    def __init__(self, role_name, system_prompt):
        self.role_name = role_name
        self.system_prompt = system_prompt
        self.history = [SystemMessage(content=system_prompt)]

    def ask(self, question):
        self.history.append(HumanMessage(content=question))
        response = llm.invoke(self.history)
        answer = response.content
        self.history.append(AIMessage(content=answer))
        return answer

    def reset(self):
        self.history = [SystemMessage(content=self.system_prompt)]


experts = [

RoleBasedAssistant(
      role_name='CEO',
      system_prompt='당신은 기업의 CEO입니다. AI 도입에 있어 비즈니스 비전, 시장 경쟁력, 그리고 전사적 전략 관점에서 의견을 제시해주세요, 3문장 한정.'
  ),
RoleBasedAssistant(
      role_name='CTO',
      system_prompt='당신은 기업의 CTO입니다. AI 도입의 기술적 타당성, 인프라 구축, 데이터 보안 및 실행 과정에서의 기술적 난제에 대해 의견을 제시해주세요., 3문장 한정'
  ),

RoleBasedAssistant(
      role_name='CFO',
      system_prompt='당신은 기업의 CFO입니다. AI 도입의 ROI(투자 대비 성과), 예산 할당, 비용 대비 편익 분석 및 재무적 리스크 관점에서 의견을 제시해주세요., 3문장 한정'
  )
]


In [11]:
question = 'AI 도입 전략에 대한 당신의 의견을 들려주세요.'
opinion = []

for expert in experts:
  answer = expert.ask(question)
  opinion.append(f"{expert.role_name}:{answer}")
  print(f"\n{expert.role_name}")
  print(f"  {answer}")


CEO
  AI 도입은 우리 비즈니스의 비전과 일치하며, 고객 경험을 혁신하고 운영 효율성을 극대화하는 데 필수적입니다. 시장 경쟁력 유지와 강화 위해 AI를 활용하여 데이터 기반 의사결정 및 개인화된 서비스 제공을 가속화해야 합니다. 나아가, 전사적 전략으로 AI 통합을 추진하여 모든 부서가 협력하고 시너지를 창출할 수 있는 생태계를 조성하는 것이 중요합니다.

CTO
  AI 도입 전략은 기업의 비즈니스 목표와 긴밀히 연결되어야 하며, 데이터 기반의 의사결정 체계를 마련하는 것이 필수적입니다. 인프라 구축 단계에서는 클라우드 서비스나 온프레미스 솔루션 중 적합한 옵션을 선택해야 하며, 데이터 보안에 대한 철저한 계획이 필요합니다. 최종적으로, 실행 과정에서 예상되는 기술적 난제를 해결하기 위해서는 지속적인 테스트와 피드백 루프를 구축하여 문제를 조기에 발견하고 수정해야 합니다.

CFO
  AI 도입 전략은 명확한 ROI 분석을 통해 비즈니스 목표와 성과를 일치시키는 것이 필수적입니다. 예산 할당 시 초기 투자 비용과 운영 비용을 고려하고, 장기적으로 비용 절감 및 생산성 향상을 기대해야 합니다. 또한, AI 도입에 따르는 재무적 리스크를 최소화하기 위해 철저한 리스크 관리 계획을 마련해야 합니다.


In [12]:
moderator = RoleBasedAssistant(
    role_name = '사회자',
    system_prompt = "당신은 경영 회의 사회자입니다. 여러 전문가 의견을 종합해 합의점을 도출해주세요. 3문장으로만 얘기하세요."
)

summary_input ="다음 의견을 종합 정리하세요:\n\n" + "\n\n".join(opinion)

moderator.ask(summary_input)

'AI 도입은 비즈니스 비전과 경쟁력 유지를 위해 필수적이며, 모든 부서가 협력하는 생태계를 조성해야 합니다. 이를 위해 데이터 기반 의사결정 체계를 구축하고, 적절한 인프라와 보안 계획을 수립해야 하며, 기술적 난제를 해결하기 위한 지속적인 피드백이 필요합니다. 마지막으로, 명확한 ROI 분석과 철저한 리스크 관리를 통해 초기 투자 및 운영 비용을 고려하여 재무적 안정성을 확보해야 합니다.'

## 사고의 사슬 (Chain of Thought)

### CoT: Chain of Thought
- Reasoning 모델과 관련
- 하나의 답변이 다음 단계의 입력으로 들어가는 연쇄적인 방식

In [13]:
# 단계별로 생각해보세요 등의 프롬프트를 이용할 때, Zero-shot 성능이 높아짐.
  # Vision Task 등의 Zero-shot과는 다름
  # Gpt 등에 요청하는걸 Zero-shot이라고 함
  # LLM에서 Shot은 '예시'의 의미임
  # Zero-shot은 예시가 없다는 말, Few-shot은 예시가 조금 있다는 뜻.

In [14]:
task_data = "2026년 우리 회사 매출: Q1: 50억, Q2: 65억, Q3: 45억, Q4: 80억"

single_prompt = f"다음 데이터를 분석하고 인사이트를 제공하세요 : {task_data}"

llm.invoke([HumanMessage(content=single_prompt)]).content

'2026년 회사의 매출 데이터를 분석해보면 다음과 같습니다:\n\n1. **총 매출**: \n   - 2026년 총 매출은 Q1(50억) + Q2(65억) + Q3(45억) + Q4(80억) = 240억 원입니다.\n\n2. **분기별 매출 추세**:\n   - **Q1**: 50억 원\n   - **Q2**: 65억 원 (전 분기 대비 +30%)\n   - **Q3**: 45억 원 (전 분기 대비 -30.8%)\n   - **Q4**: 80억 원 (전 분기 대비 +77.8%)\n\n3. **분기별 매출 성장률**:\n   - Q1에서 Q2로의 성장은 긍정적인 신호로, 제품이나 서비스에 대한 수요가 증가하고 있음을 나타냅니다.\n   - Q3의 매출 감소는 주목해야 할 상황입니다. 특히, Q2에 비해 큰 하락폭이 발생했습니다. 이러한 감소의 원인 분석이 필요합니다 (예: 계절적 요인, 경쟁 심화, 내부 문제 등).\n   - Q4에 다시 증가하여 80억 원에 도달하는 것은 회복세를 보여줍니다. 이 시기에 매출이 급증한 원인도 중요하게 분석해야 합니다 (예: 특정 프로모션, 연말 시즌 효과 등).\n\n4. **연간 매출 성장률**:\n   - 연간 매출이 240억 원이라면, 이전 연도와의 비교를 통해 성장률을 확인할 필요가 있습니다. 작년의 매출과 비교하여 성장세를 평가하면 더 명확한 인사이트를 얻을 수 있습니다.\n\n5. **분기별 비율**:\n   - Q1: 20.8%\n   - Q2: 27.1%\n   - Q3: 18.8%\n   - Q4: 33.3%\n   - Q4의 비율이 가장 높고, Q3의 비율이 가장 낮습니다. 이는 겨울철 등 특정 시기에 수요가 증가할 가능성이 있음을 나타냅니다.\n\n### 인사이트 및 제안\n1. **Q2의 성장과 Q3의 감소에 대한 분석 필요**: \n   - Q2의 성장을 지속하고 Q3에서의 감소를 방지하기 위한 전략을 마련해야 합니다. 경쟁 업체의 활동, 소비자 트렌드 변동 등을 분석하여 대응 방안을 구축할 필요가 있

In [16]:
step_prompt = f"""다음 매출 데이터를 아래 단계에 따라 분석하세요.

데이터 : {task_data}

1단계 - 데이터 정리: 분기별 매출을 표로 정리하세요.
2단계 - 추세 분석 : 전분기 대비 증감률을 계산하세요.
3단계 - 이상점 식별 : 눈에 띄는 변동이 있는 분기를 찾으세요.
4단계 - 원인 추정 : 이상점의 가능한 원인을 2가지씩 제시하세요.
5단계 - 제안: 다음 분기 전략을 1가지 제안하세요.

각 단계의 결과를 명확히 구분하여 출력하세요"""

print(llm.invoke([HumanMessage(content=step_prompt)]).content)

### 1단계 - 데이터 정리
| 분기  | 매출 (억 원) |
|-------|---------------|
| Q1    | 50            |
| Q2    | 65            |
| Q3    | 45            |
| Q4    | 80            |

### 2단계 - 추세 분석
- **Q2 대비 Q1**: (65 - 50) / 50 * 100 = 30%
- **Q3 대비 Q2**: (45 - 65) / 65 * 100 = -30.77%
- **Q4 대비 Q3**: (80 - 45) / 45 * 100 = 77.78%

| 분기  | 매출 (억 원) | 증감률 (%)  |
|-------|---------------|-------------|
| Q1    | 50            | -           |
| Q2    | 65            | 30%         |
| Q3    | 45            | -30.77%     |
| Q4    | 80            | 77.78%      |

### 3단계 - 이상점 식별
눈에 띄는 변동이 있는 분기: **Q3** (전분기 대비 -30.77% 감소)

### 4단계 - 원인 추정
**Q3 매출 감소의 가능한 원인:**
1. 시장의 시즌성 요인: 여름철 수요 감소로 인해 판매가 줄어들었을 가능성이 있음.
2. 경쟁사의 공격적인 마케팅: 다른 경쟁사가 강력한 프로모션이나 신제품 출시를 통해 고객을 빼앗았을 가능성이 있음.

### 5단계 - 제안
**다음 분기 전략 제안:**
Q4 매출 성과를 바탕으로 마케팅 캠페인을 강화하여 고객의 관심을 지속적으로 유지하며, 특히 Q3의 매출 감소 원인을 분석한 후, 시즌에 맞춘 프로모션을 개발해 여름철에도 매출을 유지하도록 적극적인 마케팅 전략을 시행하는 것이 좋습니다.


In [28]:
def build_step_prompt(task, data, steps):
  prompt_parts = [f"작업: {task}", f"\n데이터:\n{data}", "\n아래 단계를 순서대로 수행하세요:\n"]

  for i, step in enumerate(steps, 1):
    prompt_parts.append(f"{i}단계 - {step['name']}:{step['instruction']}")

  prompt_parts.append("\n각 단계의 결과를 '## N단계' 형식으로 구분하여 출력하세요.")
  return '\n'.join(prompt_parts)

In [29]:
review = "배송은 정말 빨랐어요! 그런데 상품 색상이 사진과 좀 달라서 실망했습니다."
steps = [
    {"name" : "감정 식별", "instruction" : "리뷰에서 긍정, 부정 감정을 각각 추리세요"},
    {"name" : "주제 분류", "instruction" : "언급된 주제(배송, 품질, 가격 등)를 분류하세요"},
    {"name" : "점수 산정", "instruction" : "각 주제별 만족도를 1~5점으로 평가하세요"},
    {"name" : "종합 판단", "instruction" : "전체 만족도와 재구매 가능성을 판단하세요"},
]
prompt = build_step_prompt(task = "고객 리뷰 심층 분석", data = review, steps = steps)
print(prompt)

작업: 고객 리뷰 심층 분석

데이터:
배송은 정말 빨랐어요! 그런데 상품 색상이 사진과 좀 달라서 실망했습니다.

아래 단계를 순서대로 수행하세요:

1단계 - 감정 식별:리뷰에서 긍정, 부정 감정을 각각 추리세요
2단계 - 주제 분류:언급된 주제(배송, 품질, 가격 등)를 분류하세요
3단계 - 점수 산정:각 주제별 만족도를 1~5점으로 평가하세요
4단계 - 종합 판단:전체 만족도와 재구매 가능성을 판단하세요

각 단계의 결과를 '## N단계' 형식으로 구분하여 출력하세요.


In [30]:
llm.invoke([HumanMessage(content=prompt)]).content

'## 1단계\n- 긍정 감정: "배송은 정말 빨랐어요!"\n- 부정 감정: "상품 색상이 사진과 좀 달라서 실망했습니다."\n\n## 2단계\n- 배송\n- 품질\n\n## 3단계\n- 배송 만족도: 5점 (배송이 매우 빠름)\n- 품질 만족도: 2점 (색상 불일치로 인해 실망)\n\n## 4단계\n- 전체 만족도: 3.5점 (배송은 매우 긍정적이나 품질에서 큰 실망이 있음)\n- 재구매 가능성: 낮음 (품질에 대한 실망으로 인해 재구매 의사가 낮을 것으로 판단됨)'

In [ ]:
# 실습: build_step_prompt 이용해서, "프로젝트 지연을 클라이언트에게 알리는 이메일", "신규 파트너십 제안 이메일" 작성

In [31]:
def build_step_prompt(task, data, steps):
  prompt_parts = [f"작업: {task}", f"\n데이터:\n{data}", "\n아래 단계를 순서대로 수행하세요:\n"]

  for i, step in enumerate(steps, 1):
    prompt_parts.append(f"{i}단계 - {step['name']}:{step['instruction']}")

  prompt_parts.append("\n각 단계의 결과를 '## N단계' 형식으로 구분하여 출력하세요.")
  return '\n'.join(prompt_parts)

In [ ]:
review = "배송은 정말 빨랐어요! 그런데 상품 색상이 사진과 좀 달라서 실망했습니다."
steps = [
    {"name" : "감정 식별", "instruction" : "리뷰에서 긍정, 부정 감정을 각각 추리세요"},
    {"name" : "주제 분류", "instruction" : "언급된 주제(배송, 품질, 가격 등)를 분류하세요"},
    {"name" : "점수 산정", "instruction" : "각 주제별 만족도를 1~5점으로 평가하세요"},
    {"name" : "종합 판단", "instruction" : "전체 만족도와 재구매 가능성을 판단하세요"},
]

In [ ]:
email_steps = [ # name을 어떤 단계별 목차 등으로 구성하는듯
    {'name': '상황 분석', 'instruction'}

]

### 출력 형식 지정
- prompt formatting
- https://arxiv.org/pdf/2411.10541
- https://arxiv.org/pdf/2408.02442

In [ ]:
# 답변성능
# 정답: I go to school / 모델 출력 : I go school (몇개가 match 되는지 등의 metric)
# 정량평가에 대한 metric도 중요하고, 사람 관점에서 정성적인 평가도 중요.

# 두번쨰 논문
  # 답변 형식을 json, xml 등으로 내면 성능이 좋다 vs 나쁘다(추론 능력이 떨어진다.)
  # 억지로 구조화된 출력형식 -> 추론보다 출력 포맷을 맞추는데 연산자원을 낭비하게 됨.